In [1]:
# HealthGuard AI - Stroke Data Cleaning
# Author: HealthGuard AI Team
# Date: 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
%matplotlib inline

plt.style.use('seaborn-v0_8')

print("=" * 50)
print("  HealthGuard AI - Stroke Data Cleaning")
print("=" * 50)
print("Libraries Loaded Successfully!")

  HealthGuard AI - Stroke Data Cleaning
Libraries Loaded Successfully!


In [2]:
# Load Stroke Dataset

df = pd.read_csv("E:/HealthGuard_AI/data/raw/stroke.csv")

print(f"Original Shape: {df.shape}")
print(f"Missing Values:")
print(df.isnull().sum())
df.head()

Original Shape: (5110, 12)
Missing Values:
id                     0
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


In [3]:
# Step 1: Drop ID Column
# ID column not needed for prediction

print("=" * 50)
print("STEP 1: DROP ID COLUMN")
print("=" * 50)

df = df.drop('id', axis=1)
print(f"Shape after dropping id: {df.shape}")
print("ID column removed!")

STEP 1: DROP ID COLUMN
Shape after dropping id: (5110, 11)
ID column removed!


In [4]:
# Step 2: Handle Missing Values
# BMI has missing values - fill with median

print("=" * 50)
print("STEP 2: HANDLE MISSING VALUES")
print("=" * 50)

print("Before:")
print(df.isnull().sum())

# BMI - fill with median grouped by stroke
df['bmi'] = df.groupby('stroke')['bmi'].transform(
    lambda x: x.fillna(x.median()))

# If still any missing
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

print("\nAfter:")
print(df.isnull().sum())
print("Missing Values Fixed!")

STEP 2: HANDLE MISSING VALUES
Before:
gender                 0
age                    0
hypertension           0
heart_disease          0
ever_married           0
work_type              0
Residence_type         0
avg_glucose_level      0
bmi                  201
smoking_status         0
stroke                 0
dtype: int64

After:
gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
stroke               0
dtype: int64
Missing Values Fixed!


In [5]:
# Step 3: Remove Duplicates

print("=" * 50)
print("STEP 3: REMOVE DUPLICATES")
print("=" * 50)

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicates")

STEP 3: REMOVE DUPLICATES
Before: 5110 rows
After: 5110 rows
Removed: 0 duplicates


In [6]:
# Step 4: Handle Unknown Values in smoking_status

print("=" * 50)
print("STEP 4: HANDLE UNKNOWN VALUES")
print("=" * 50)

print("Smoking Status Values:")
print(df['smoking_status'].value_counts())

# Replace Unknown with most common value
most_common = df[df['smoking_status'] != 'Unknown'][
    'smoking_status'].mode()[0]
df['smoking_status'] = df['smoking_status'].replace(
    'Unknown', most_common)

print("\nAfter fixing Unknown:")
print(df['smoking_status'].value_counts())

STEP 4: HANDLE UNKNOWN VALUES
Smoking Status Values:
smoking_status
never smoked       1892
Unknown            1544
formerly smoked     885
smokes              789
Name: count, dtype: int64

After fixing Unknown:
smoking_status
never smoked       3436
formerly smoked     885
smokes              789
Name: count, dtype: int64


In [7]:
# Step 5: Encode Categorical Features

print("=" * 50)
print("STEP 5: ENCODE CATEGORICAL FEATURES")
print("=" * 50)

le = LabelEncoder()

cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f" {col} encoded")

print("\nEncoding Complete!")

STEP 5: ENCODE CATEGORICAL FEATURES
 gender encoded
 ever_married encoded
 work_type encoded
 Residence_type encoded
 smoking_status encoded

Encoding Complete!


In [8]:
# Step 6: Remove Outliers

print("=" * 50)
print("STEP 6: REMOVE OUTLIERS")
print("=" * 50)

before = len(df)

outlier_cols = ['age', 'avg_glucose_level', 'bmi']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers removed")

    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f"\nBefore: {before} rows")
print(f"After: {after} rows")

STEP 6: REMOVE OUTLIERS
age: 0 outliers removed
avg_glucose_level: 627 outliers removed
bmi: 100 outliers removed

Before: 5110 rows
After: 4383 rows


In [9]:
# Step 7: Feature Scaling

print("=" * 50)
print("STEP 7: FEATURE SCALING")
print("=" * 50)

X = df.drop('stroke', axis=1)
y = df['stroke']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaling Complete!")
print(f"Features: {X_scaled.shape}")

STEP 7: FEATURE SCALING
Scaling Complete!
Features: (4383, 10)


In [10]:
# Train Test Split FIRST, then SMOTE only on Training data
# Critical for Stroke since original imbalance is extreme (95/5)

print("=" * 50)
print("TRAIN TEST SPLIT (BEFORE SMOTE)")
print("=" * 50)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Train (before SMOTE): {len(X_train_raw)}")
print(f"Test (untouched, real-world distribution): {len(X_test)}")

print("\n" + "=" * 50)
print("SMOTE ON TRAINING DATA ONLY")
print("=" * 50)

print("Before SMOTE:")
print(f"No Stroke (0): {(y_train_raw==0).sum()}")
print(f"Stroke (1): {(y_train_raw==1).sum()}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

print("\nAfter SMOTE:")
print(f"No Stroke (0): {(y_train==0).sum()}")
print(f"Stroke (1): {(y_train==1).sum()}")
print(f"Final Training: {len(X_train)}, Final Testing: {len(X_test)}")

TRAIN TEST SPLIT (BEFORE SMOTE)
Train (before SMOTE): 3506
Test (untouched, real-world distribution): 877

SMOTE ON TRAINING DATA ONLY
Before SMOTE:
No Stroke (0): 3375
Stroke (1): 131

After SMOTE:
No Stroke (0): 3375
Stroke (1): 3375
Final Training: 6750, Final Testing: 877


In [11]:
# Step 10: Save Cleaned Data

print("=" * 50)
print("STEP 10: SAVE CLEANED DATA")
print("=" * 50)

df_cleaned = pd.concat([X_scaled,
                        y.reset_index(drop=True)], axis=1)
df_cleaned.to_csv(
    "E:/HealthGuard_AI/data/processed/stroke_cleaned_final.csv",
    index=False)

X_train.to_csv(
    "E:/HealthGuard_AI/data/processed/stroke_X_train.csv",
    index=False)
X_test.to_csv(
    "E:/HealthGuard_AI/data/processed/stroke_X_test.csv",
    index=False)
y_train.to_csv(
    "E:/HealthGuard_AI/data/processed/stroke_y_train.csv",
    index=False)
y_test.to_csv(
    "E:/HealthGuard_AI/data/processed/stroke_y_test.csv",
    index=False)

print("Files Saved:")
print(" stroke_cleaned_final.csv")
print(" stroke_X_train.csv")
print(" stroke_X_test.csv")
print(" stroke_y_train.csv")
print(" stroke_y_test.csv")

STEP 10: SAVE CLEANED DATA
Files Saved:
 stroke_cleaned_final.csv
 stroke_X_train.csv
 stroke_X_test.csv
 stroke_y_train.csv
 stroke_y_test.csv


In [12]:
# Cleaning Summary

print("=" * 60)
print("   STROKE CLEANING - SUMMARY REPORT")
print("=" * 60)

print("\nTECHNIQUES USED:")
print("-" * 40)
print("1. ID Column Removal")
print("2. BMI Missing Values - Grouped Median")
print("3. Duplicate Removal")
print("4. Unknown Values Treatment")
print("5. Label Encoding")
print("6. IQR Outlier Removal")
print("7. Standard Scaling")
print("8. Train Test Split BEFORE SMOTE (leakage-free)")
print("9. SMOTE - Applied only on Training Data")

print(f"\nOriginal Rows: 5110")
print(f"After Cleaning: {len(df)}")
print(f"Training Set (after SMOTE): {len(X_train)}")
print(f"Testing Set (untouched, real): {len(X_test)}")

print("\nStroke Cleaning Complete!")
print("=" * 60)

   STROKE CLEANING - SUMMARY REPORT

TECHNIQUES USED:
----------------------------------------
1. ID Column Removal
2. BMI Missing Values - Grouped Median
3. Duplicate Removal
4. Unknown Values Treatment
5. Label Encoding
6. IQR Outlier Removal
7. Standard Scaling
8. Train Test Split BEFORE SMOTE (leakage-free)
9. SMOTE - Applied only on Training Data

Original Rows: 5110
After Cleaning: 4383
Training Set (after SMOTE): 6750
Testing Set (untouched, real): 877

Stroke Cleaning Complete!


In [13]:
# Save Scaler for Web App
import pickle

scaler_path = "E:/HealthGuard_AI/models/saved/stroke_scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(" Stroke Scaler Saved!")
print(f"Location: {scaler_path}")

 Stroke Scaler Saved!
Location: E:/HealthGuard_AI/models/saved/stroke_scaler.pkl


In [14]:
# FIXED: Pipeline on RAW unscaled data
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import pickle
import pandas as pd

# Reload RAW cleaned data (before scaling)
df_raw = pd.read_csv("E:/HealthGuard_AI/data/raw/stroke.csv")
df_raw = df_raw.drop('id', axis=1)
df_raw['bmi'] = df_raw['bmi'].fillna(df_raw['bmi'].median())

le = LabelEncoder()
cat_cols = df_raw.select_dtypes(include=['object']).columns
for col in cat_cols:
    df_raw[col] = le.fit_transform(df_raw[col].astype(str))

X_raw = df_raw.drop('stroke', axis=1)
y_raw = df_raw['stroke']

# SMOTE on raw data
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_raw, y_raw)

# Split
X_tr, X_te, y_tr, y_te = train_test_split(
    X_resampled, y_resampled,
    test_size=0.2, random_state=42,
    stratify=y_resampled)

feature_names = X_raw.columns.tolist()
print("Stroke Features:", feature_names)

# Pipeline handles scaling
stroke_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100, random_state=42, max_depth=5))
])

stroke_pipeline.fit(X_tr, y_tr)
y_pred = stroke_pipeline.predict(X_te)
print(f"Accuracy: {accuracy_score(y_te, y_pred):.4f}")

# High risk test
test_sample = pd.DataFrame([[
    1, 55, 1, 1, 1, 2, 1, 250, 31, 2
]], columns=feature_names)
prob = stroke_pipeline.predict_proba(test_sample)[0][1]
print(f"High Risk Test: {prob * 100:.2f}%")

pipeline_path = "E:/HealthGuard_AI/models/saved/stroke_pipeline.pkl"
with open(pipeline_path, 'wb') as f:
    pickle.dump(stroke_pipeline, f)
print("Stroke Pipeline Saved!")

Stroke Features: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']
Accuracy: 0.8344
High Risk Test: 40.50%
Stroke Pipeline Saved!
